# AI Loan Advisory Chatbot
## Phase 1: PDF Processing Pipeline

### Objective

This notebook is responsible for processing all official banking PDF documents.

It performs the following tasks:

- Read PDF files
- Extract text
- Clean extracted text
- Generate metadata
- Save processed outputs

The processed data generated in this notebook will be used in the subsequent stages:
- Intelligent Chunking
- Embedding Generation
- FAISS Vector Database
- Retrieval-Augmented Generation (RAG)

## Why PDF Processing?

Large Language Models cannot directly understand PDF files.

Before creating embeddings, every document must be converted into clean text.

This notebook converts banking PDFs into a standardized format that can later be used for semantic search and question answering.

Pipeline:

Official PDFs
      ↓
Text Extraction
      ↓
Cleaning
      ↓
Metadata
      ↓
Ready for Chunking

In [1]:
!pip install -q pymupdf pdfplumber pytesseract

import fitz
import pdfplumber
from pathlib import Path
import json
import re
import os
from datetime import datetime
import pytesseract
from PIL import Image
import io

from google.colab import drive
drive.mount('/content/drive')

# Setup Directories
PROJECT_ROOT = Path("/content/drive/MyDrive/AI_Loan_Advisory_Chatbot")
DATASET_DIR = PROJECT_ROOT / "dataset"
PROCESSED_DIR = PROJECT_ROOT / "processed"
CLEAN_TEXT_DIR = PROCESSED_DIR / "cleaned_text"
METADATA_DIR = PROCESSED_DIR / "metadata"
CHUNKS_DIR = PROCESSED_DIR / "chunks"
VECTOR_DB_DIR = PROJECT_ROOT / "vector_db"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

folders = [CLEAN_TEXT_DIR, METADATA_DIR, CHUNKS_DIR, VECTOR_DB_DIR, MODELS_DIR, OUTPUTS_DIR]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Project structure is ready!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 98.3 MB/s eta 0:00:00
Mounted at /content/drive
Project structure is ready!


In [2]:
def fix_encoding(text):
    """Fix common encoding artifacts found in PDFs."""
    replacements = {"ï¿½": "'", "â€™": "'", "â€œ": '"', "â€ ": '"', "â€“": "-", "â€”": "-", "Â": ""}
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def normalize_whitespace(text):
    """Normalize spaces, tabs and newlines."""
    text = text.replace("\r", "\n").replace("\t", " ")
    text = re.sub(r" +", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def remove_ui_noise(text):
    """Remove common website UI elements."""
    noise_phrases = ["Accessibility Menu", "Color Adjustment", "Virtual Keyboard", "Hover on content",
                     "Voice/Accent", "Page Structure", "Profiles", "Content", "Colors", "Volume",
                     "Rate", "Pitch", "Apply Now", "opens in a new tab"]
    for phrase in noise_phrases:
        text = text.replace(phrase, "")
    return text

def clean_text(text):
    """Clean text without destroying legitimate duplicate lines."""
    text = fix_encoding(text)
    text = normalize_whitespace(text)
    text = remove_ui_noise(text)

    cleaned = []
    for line in text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            cleaned.append(line)
    return "\n".join(cleaned)

def detect_document_type(filename):
    """Detect the document type from the filename."""
    filename = filename.lower()
    if "faq" in filename: return "FAQ"
    elif "mitc" in filename: return "MITC"
    elif "term" in filename or "condition" in filename: return "Terms & Conditions"
    elif "interest" in filename: return "Interest Rates"
    else: return "Overview"

def extract_metadata_from_path(pdf_path):
    """Extract metadata dynamically regardless of folder depth."""
    relative_path = pdf_path.relative_to(DATASET_DIR)
    parts = relative_path.parts

    return {
        "bank": parts[0] if len(parts) > 0 else "Unknown",
        "loan_type": parts[1] if len(parts) > 1 else "Unknown",
        "document_type": detect_document_type(pdf_path.name),
        "file_name": pdf_path.name,
        "file_path": str(relative_path),
        "extension": pdf_path.suffix,
        "processed_date": datetime.now().strftime("%Y-%m-%d")
    }

def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF keeping page mappings and handling corrupted PDFs."""
    try:
        document = fitz.open(pdf_path)
        if document.needs_pass:
             return {"pages": [], "page_count": 0, "character_count": 0, "word_count": 0, "error": "Password protected"}

        pages_data = []
        total_chars = 0
        total_words = 0

        for page_num in range(len(document)):
            page = document[page_num]
            page_text = page.get_text().strip()

            # OCR ONLY if native extraction yields very little text
            if len(page_text.split()) < 20:
                try:
                    pix = page.get_pixmap(matrix=fitz.Matrix(3, 3))
                    image = Image.open(io.BytesIO(pix.tobytes("png")))
                    ocr_text = pytesseract.image_to_string(image, config="--oem 3 --psm 6").strip()
                    page_text = ocr_text if ocr_text else page_text
                except Exception as e:
                    print(f"OCR failed on {pdf_path.name} page {page_num}: {e}")

            cleaned_page_text = clean_text(page_text)
            pages_data.append({"page_number": page_num + 1, "text": cleaned_page_text})

            total_chars += len(cleaned_page_text)
            total_words += len(cleaned_page_text.split())

        return {"pages": pages_data, "page_count": len(document), "character_count": total_chars, "word_count": total_words}

    except Exception as e:
        return {"pages": [], "page_count": 0, "character_count": 0, "word_count": 0, "error": str(e)}
    finally:
        if 'document' in locals():
            document.close()

def process_pdf(pdf_path):
    """Complete processing pipeline for a single PDF adapted for page mappings."""
    metadata = extract_metadata_from_path(pdf_path)
    extracted = extract_text_from_pdf(pdf_path)

    if "error" in extracted:
         return {"metadata": metadata, "error": extracted["error"]}

    raw_combined = "\n\n".join([p["text"] for p in extracted.get("pages", [])])
    cleaned_combined = clean_text(raw_combined)

    return {
        "metadata": metadata,
        "statistics": {"page_count": extracted.get("page_count", 0), "character_count": extracted.get("character_count", 0), "word_count": extracted.get("word_count", 0)},
        "content": {"pages": extracted.get("pages", []), "raw_text": raw_combined, "cleaned_text": cleaned_combined}
    }

def save_document(document):
    """Save cleaned text and metadata while preserving the dataset folder structure."""
    if "error" in document:
        return None, None

    metadata = document["metadata"]
    bank = metadata["bank"]
    loan = metadata["loan_type"]

    text_folder = CLEAN_TEXT_DIR / bank / loan
    meta_folder = METADATA_DIR / bank / loan
    text_folder.mkdir(parents=True, exist_ok=True)
    meta_folder.mkdir(parents=True, exist_ok=True)

    filename = Path(metadata["file_path"]).stem
    text_file = text_folder / f"{filename}.txt"
    with open(text_file, "w", encoding="utf-8") as f:
        f.write(document["content"]["cleaned_text"])

    metadata_file = meta_folder / f"{filename}.json"
    with open(metadata_file, "w", encoding='utf-8') as f:
        json.dump({**document["metadata"], **document["statistics"]}, f, indent=4)

    return text_file, metadata_file

In [3]:
pdf_files = sorted(DATASET_DIR.rglob("*.pdf"))
print(f"Total PDFs Found: {len(pdf_files)}\n")

processed_documents = []
failed_documents = []

for pdf in pdf_files:
    try:
        document = process_pdf(pdf)
        if "error" in document:
            failed_documents.append({"file": pdf.name, "error": document["error"]})
            print(f"❌ Failed: {pdf.name} | Error: {document['error']}")
        else:
            save_document(document)
            processed_documents.append(document)
            print(f"✅ Processed: {pdf.name}")
    except Exception as e:
        failed_documents.append({"file": pdf.name, "error": str(e)})
        print(f"❌ Critical Failure: {pdf.name} | Error: {e}")

# Print Summary
total_pages = sum(doc["statistics"]["page_count"] for doc in processed_documents)
total_words = sum(doc["statistics"]["word_count"] for doc in processed_documents)
total_characters = sum(doc["statistics"]["character_count"] for doc in processed_documents)

print("\n" + "=" * 70)
print("AI LOAN ADVISORY CHATBOT - PDF PROCESSING SUMMARY")
print("=" * 70)
print(f"Total PDFs Found          : {len(pdf_files)}")
print(f"Successfully Processed    : {len(processed_documents)}")
print(f"Failed                    : {len(failed_documents)}")
print("-" * 70)
print(f"Total Pages               : {total_pages}")
print(f"Total Words               : {total_words:,}")
print(f"Total Characters          : {total_characters:,}")
print("=" * 70)

Total PDFs Found: 36

✅ Processed: HDFC_Car_Loan_MITC.pdf
✅ Processed: HDFC_Car_Loan_Overview.pdf
✅ Processed: HDFC_Home_Loan_MITC.pdf
✅ Processed: HDFC_Home_Loan_Overview.pdf
✅ Processed: HDFC_Personal_Loan_Overview.pdf
✅ Processed: SBI_EDUCATION_LOAN_Main_Page.pdf
✅ Processed: SBI_EDUCATION_LOAN_Scholar_Loan..pdf
✅ Processed: SBI_EDUCATION_LOAN_Student_Loan.pdf
✅ Processed: SBI_Education_Loan_Interest_Rate.pdf
✅ Processed: SBI_Education_MITC_Scholar_Loan.pdf
✅ Processed: SBI_Educational_Loan_Faq's.pdf
✅ Processed: SBI_GOLD_LOAN_Product_Overview.pdf.pdf
✅ Processed: SBI_Home_Loan_Agreement.pdf
✅ Processed: SBI_Home_Loan_Application_Form.pdf
✅ Processed: SBI_Home_Loan_Documents_Required.pdf
✅ Processed: SBI_Home_Loan_EMI_Calculator.pdf
✅ Processed: SBI_Home_Loan_Eligibility.pdf
✅ Processed: SBI_Home_Loan_FAQs.pdf
✅ Processed: SBI_Home_Loan_Features.pdf
✅ Processed: SBI_Home_Loan_Interest_Rates.pdf
✅ Processed: SBI_Home_Loan_Interest_Rates_and_Fees.pdf
✅ Processed: SBI_Home_Loan_Process